In [0]:
from datetime import datetime
from decimal import Decimal

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    DecimalType
)

from delta.tables import DeltaTable


# ============================================================
# 1. CONFIGURATION
# ============================================================

storage_account = "stsupplychain2026xx"

silver_orders_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "silver/orders/"
)


# ============================================================
# 2. ÉTAT DE LA TABLE AVANT MERGE
# ============================================================

silver_orders_before = (
    spark.read
    .format("delta")
    .load(silver_orders_path)
)

count_before = silver_orders_before.count()

print(
    "Nombre de lignes AVANT MERGE :",
    count_before
)

print("\n=== O0131 AVANT MERGE ===")

display(
    silver_orders_before
    .filter(
        F.col("order_id") == "O0131"
    )
    .select(
        "order_id",
        "customer_id",
        "total_amount",
        "currency",
        "order_status",
        "updated_at",
        "batch_id"
    )
)


# ============================================================
# 3. MINI-BATCH DE TEST
# ============================================================

merge_test_data = [

    # --------------------------------------------------------
    # CAS 1 : nouvelle commande
    # O0135 n'existe pas encore
    # => INSERT
    # --------------------------------------------------------
    (
        "O0135",
        "CUST0020",
        datetime(2026, 1, 12, 10, 0),
        "Rabat",
        "Morocco",
        Decimal("1450.00"),
        "MAD",
        "CONFIRMED",
        datetime(2026, 1, 12, 10, 0)
    ),

    # --------------------------------------------------------
    # CAS 2 : correction d'une commande existante
    # O0131 existe déjà
    # updated_at est plus récent
    # => UPDATE
    # --------------------------------------------------------
    (
        "O0131",
        "CUST0010",
        datetime(2026, 1, 10, 8, 0),
        "Rabat",
        "Morocco",
        Decimal("1500.00"),
        "MAD",
        "PROCESSING",
        datetime(2026, 1, 12, 11, 0)
    )
]


# ============================================================
# 4. SCHÉMA DU MINI-BATCH
# ============================================================

merge_test_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("order_time", TimestampType(), False),
    StructField("destination_city", StringType(), False),
    StructField("destination_country", StringType(), False),
    StructField("total_amount", DecimalType(10, 2), False),
    StructField("currency", StringType(), False),
    StructField("order_status", StringType(), False),
    StructField("updated_at", TimestampType(), False)
])


# ============================================================
# 5. CRÉATION DU DATAFRAME SOURCE
# ============================================================

merge_source = (
    spark.createDataFrame(
        merge_test_data,
        schema=merge_test_schema
    )

    # Métadonnées techniques pour garder un schéma
    # compatible avec silver_orders
    .withColumn(
        "source_file",
        F.lit("phase4_merge_test")
    )

    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )

    .withColumn(
        "batch_id",
        F.lit("merge_test_001")
    )
)

print("\n=== SOURCE DU MERGE ===")

display(
    merge_source
    .orderBy("order_id")
)


# ============================================================
# 6. RÉCUPÉRER LA TABLE DELTA CIBLE
# ============================================================

silver_orders_delta = (
    DeltaTable.forPath(
        spark,
        silver_orders_path
    )
)


# ============================================================
# 7. DELTA MERGE
# ============================================================

(
    silver_orders_delta.alias("target")

    .merge(
        merge_source.alias("source"),
        "target.order_id = source.order_id"
    )

    # Une ligne existe déjà.
    # On la met à jour SEULEMENT si la source
    # possède une version plus récente.
    .whenMatchedUpdateAll(
        condition="""
            source.updated_at > target.updated_at
        """
    )

    # La clé métier n'existe pas encore.
    .whenNotMatchedInsertAll()

    .execute()
)

print("\nMERGE terminé.")


# ============================================================
# 8. ÉTAT APRÈS MERGE
# ============================================================

silver_orders_after = (
    spark.read
    .format("delta")
    .load(silver_orders_path)
)

count_after = silver_orders_after.count()

print(
    "Nombre de lignes AVANT MERGE :",
    count_before
)

print(
    "Nombre de lignes APRÈS MERGE :",
    count_after
)

print(
    "Différence :",
    count_after - count_before
)


# ============================================================
# 9. VÉRIFIER L'INSERT
# ============================================================

print("\n=== O0135 : INSERT ===")

display(
    silver_orders_after
    .filter(
        F.col("order_id") == "O0135"
    )
    .select(
        "order_id",
        "customer_id",
        "total_amount",
        "currency",
        "order_status",
        "updated_at",
        "batch_id"
    )
)


# ============================================================
# 10. VÉRIFIER L'UPDATE
# ============================================================

print("\n=== O0131 : UPDATE ===")

display(
    silver_orders_after
    .filter(
        F.col("order_id") == "O0131"
    )
    .select(
        "order_id",
        "customer_id",
        "total_amount",
        "currency",
        "order_status",
        "updated_at",
        "batch_id"
    )
)